# Did LEVIATHAN filter the inversions?
So it doesn't seem like repeat-regions are too responsible for the false negatives (uncalled inversions). The next place to look is the `.candidates` files LEVIATHAN outputs. These files contain the list of all putatively called structural variants, which the program then filters based on some criteria to output the final called variant set. The question we're trying to address here is: **did LEVIATHAN identify the inversions initially and then filter them out?**

## Collapse the candidates
First, let's parse through all the candidates files to create a dataframe that we can then use for assessing what's going on.

In [ ]:
sizes = ["small" "medium" "large" "xl"]
depths = ["0.5" "2" "5" "10" "20"]
logdir = "logs/leviathan"

with open("simulated_data/called_sv/leviathan/all.candidates", "w") as candidates:
  for _size in sizes:
    for _depth in depths:
      _dir = f"simulated_data/called_sv/leviathan/{_size}/depth_{_depth}"
      # GET THE LIST OF .candidates FILES IN _dir/by_sample90/logdir
      _cand_files = []
      # append to the list the .candidates file in _dir/by_pop90/logdir/pop1.candidates
      _cand_files.append(f"{_dir}/by_pop90/{logdir}/pop1.candidates")
      # iterate through file list
      for i in _cand_files:
        pass

We'll need to assess performance, so we first need to take the "truth" set (the inversions we deliberately simulated) for each individual and merge it into a single file for every treatment. In other words, each individual has a VCF of inversions for each haplotype that was used for simulation, so we need to combine them such that a single VCF represents all the inversions an individual has (regardless of haplotype).

This also requires adding the contig info to the VCF's, which simuG didn't add unfortunately. We will overwrite the VCF files with a modified version that includes the contig info below the `##source=simuG.pl` line:

```
##contig=<ID=2L,length=23513712>
##contig=<ID=2R,length=25286936>
##contig=<ID=3L,length=28110227>
##contig=<ID=3R,length=32079331>
```

This will ultimately require us to use `bcftools merge`, which also means the VCF files need to be converted to BCF or VCF.GZ format
We also need to fix a typo from `simuG` that sets an incorrect type for the field `INFO/END` to `Type=Number`

In [11]:
import os

contig_text = """\
##contig=<ID=2L,length=23513712>
##contig=<ID=2R,length=25286936>
##contig=<ID=3L,length=28110227>
##contig=<ID=3R,length=32079331>\
"""

for _size in ["small", "medium", "large", "xl"]:
    for _sample in range(1,11):
        SAMPLE = "sample_" + str(_sample).zfill(2)
        DIR = f"simulated_variants/inversions_genomes/{_size}/samples/{SAMPLE}"
        for _hap in [1,2]:
            BASENAME = f"{DIR}/{SAMPLE}.{_size}.hap"
            with open(f"{BASENAME}{_hap}.vcf", "r") as vcf, open(f"{BASENAME}{_hap}.tmp.vcf", "w") as vcf_out:
                for _vcf in vcf:
                    newline = _vcf.replace("##source=simuG.pl", contig_text).replace("##INFO=<ID=END,Number=1,Type=String,", "##INFO=<ID=END,Number=1,Type=Integer,")
                    vcf_out.write(newline)
                    #vcf_out.write(_vcf.replace("##source=simuG.pl", contig_text))
                    #vcf_out.write(_vcf.replace("##INFO=<ID=END,Number=1,Type=String,", "##INFO=<ID=END,Number=1,Type=Number,"))
            os.system(f"/programs/bcftools-1.20/bin/bcftools view -Oz -o {BASENAME}{_hap}.vcf.gz --write-index=tbi {BASENAME}{_hap}.tmp.vcf")
            os.remove(f"{BASENAME}{_hap}.tmp.vcf")



Now that they follow proper VCF specification, we can merge them with `bcftools merge` and remove the intermediate files

In [12]:
for _size in ["small", "medium", "large", "xl"]:
    for _sample in range(1,11):
        SAMPLE = "sample_" + str(_sample).zfill(2)
        DIR = f"simulated_variants/inversions_genomes/{_size}/samples/{SAMPLE}"
        BASENAME = f"{DIR}/{SAMPLE}.{_size}"
        os.system(f"/programs/bcftools-1.20/bin/bcftools merge -Oz -o {BASENAME}.vcf.gz --write-index=tbi {BASENAME}.hap1.vcf.gz {BASENAME}.hap2.vcf.gz")
        for i in [1,2]:
            os.remove(f"{BASENAME}.hap{i}.vcf.gz")
            os.remove(f"{BASENAME}.hap{i}.vcf.gz.tbi")
        